<a href="https://colab.research.google.com/github/edmilsondejesus/scrapy_extract/blob/main/Script_BERTimbau_github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#pip install transformers

In [ ]:
#pip install torch

In [ ]:
#pip install --upgrade accelerate


In [ ]:
#pip install --upgrade transformers[torch]

In [ ]:
# ============================================
# AP5 - Modelos de Linguagem com BERTimbau
# Versão compatível com versões anteriores do transformers
# ============================================

import torch
import numpy as np
from transformers import BertTokenizer, BertForMaskedLM, BertForSequenceClassification, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from torch.utils.data import Dataset
import json
import re
import plotly.graph_objects as go
from collections import Counter
import os
import sys

In [ ]:
# ============================================
# 1. CARREGAR CORPUS DO JSON (Simplificado)
# ============================================
print("="*60)
print("1. CARREGANDO CORPUS DO JSON")
print("="*60)

import os
import sys
import json

# Verificar se está no Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Tentar carregar o arquivo
json_path = "stil2023_articles_limpo.json"

if not os.path.exists(json_path):
    # Procurar outros JSONs
    arquivos_json = [f for f in os.listdir('.') if f.endswith('.json')]
    if arquivos_json:
        json_path = arquivos_json[0]
        print(f" Usando: {json_path}")
    elif IN_COLAB:
        print(" Faça upload do arquivo JSON:")
        uploaded = files.upload()
        json_path = next(iter(uploaded.keys()))
        print(f" Arquivo carregado: {json_path}")
    else:
        print(" Arquivo JSON não encontrado!")
        sys.exit(1)
else:
    print(f" Arquivo encontrado: {json_path}")

# Carregar
with open(json_path, encoding='utf-8') as f:
    articles = json.load(f)

print(f" Artigos carregados: {len(articles)}")

1. CARREGANDO CORPUS DO JSON
 Faça upload do arquivo JSON:


Saving stil2023_articles_limpo.json to stil2023_articles_limpo.json
 Arquivo carregado: stil2023_articles_limpo.json
 Artigos carregados: 30


In [ ]:
# ============================================
# 2. CARREGAR BERTIMBAU
# ============================================
print("\n" + "="*60)
print("2. CARREGANDO BERTIMBAU")
print("="*60)

tokenizer = BertTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')
model_base = BertForMaskedLM.from_pretrained('neuralmind/bert-base-portuguese-cased')
print("BERTimbau base carregado")



2. CARREGANDO BERTIMBAU


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTimbau base carregado


In [ ]:
# ============================================
# 3. EXTRAIR TEXTOS E TOKENS
# ============================================
print("\n" + "="*60)
print("3. EXTRAINDO TEXTOS E TOKENS")
print("="*60)

textos_artigos = []
todos_tokens = []

for artigo in articles:
    texto = artigo.get("artigo_completo", "")
    if texto and len(texto) > 100:
        textos_artigos.append(texto)

    tokens = artigo.get("artigo_tokenizado", [])
    for token in tokens:
        #if re.match(r'[a-zA-Záéíóúãõâêôç]+', token) and len(token) > 2:
        todos_tokens.append(token)

freq_tokens = Counter(todos_tokens) # Conta quantas vezes cada palavra aparece
print(f"Total de artigos: {len(textos_artigos)}")
print(f"Total de tokens únicos: {len(freq_tokens)}")


3. EXTRAINDO TEXTOS E TOKENS
Total de artigos: 30
Total de tokens únicos: 12782


In [ ]:
# ============================================
# 4. ENCONTRAR SUBSTANTIVO E VERBO MAIS FREQUENTES
# ============================================
print("\n" + "="*60)
print("4. ANALISANDO POS TAGS")
print("="*60)

substantivos = []
verbos = []

for artigo in articles:
    tokens = artigo.get("artigo_tokenizado", [])
    pos_tags = artigo.get("pos_tagger", [])
    for token, pos in zip(tokens, pos_tags):
        if pos == 'NOUN' and len(token) > 2:
            substantivos.append(token)
        elif pos == 'VERB' and len(token) > 2:
            verbos.append(token)

freq_subst = Counter(substantivos)
freq_verb = Counter(verbos)

substantivo_top = freq_subst.most_common(1)[0][0] if substantivos else "dados"
verbo_top = freq_verb.most_common(1)[0][0] if verbos else "pode"

print(f"Substantivo mais frequente: '{substantivo_top}' ({freq_subst[substantivo_top]} ocorrências)")
print(f"Verbo mais frequente: '{verbo_top}' ({freq_verb[verbo_top]} ocorrências)")


4. ANALISANDO POS TAGS
Substantivo mais frequente: 'anotacão' (235 ocorrências)
Verbo mais frequente: 'pode' (125 ocorrências)


In [ ]:
# ============================================
# 5. DATASET PARA FINE-TUNING
# ============================================
print("\n" + "="*60)
print("5. PREPARANDO DATASET PARA FINE-TUNING")
print("="*60)

class TextDataset(Dataset):
    def __init__(self, textos, tokenizer, max_length=128):
        self.textos = textos
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, idx):
        texto = self.textos[idx]
        encoding = self.tokenizer(
            texto,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze()
        }

if len(textos_artigos) > 0:
    dataset_ft = TextDataset(textos_artigos, tokenizer, max_length=256)

    # Data collator para MLM
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=0.15
    )

    # Carregar modelo para fine-tuning
    model_ft = BertForMaskedLM.from_pretrained('neuralmind/bert-base-portuguese-cased')

    # Configurar treinamento
    training_args = TrainingArguments(
        output_dir="./bertimbau_finetuned",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        save_steps=500,
        save_total_limit=2,
        logging_steps=50,
        report_to="none"
    )

    trainer = Trainer(
        model=model_ft,
        args=training_args,
        data_collator=data_collator,
        train_dataset=dataset_ft,
    )

    print("Fine-tuning configurado!")
    print(f"  Dataset: {len(dataset_ft)} textos")
    trainer.train()  # fine-tuning do Dataset do corpus
else:
    print("ATENÇÃO: Nenhum texto válido para fine-tuning!")
    model_ft = model_base


5. PREPARANDO DATASET PARA FINE-TUNING


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fine-tuning configurado!
  Dataset: 30 textos


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# ============================================
# 6. FUNÇÕES PARA OBTER VETORES
# ============================================
print("\n" + "="*60)
print("6. FUNÇÕES PARA EXTRAIR VETORES")
print("="*60)

def obter_vetor_base(palavra):
    """Vetor usando BERTimbau base"""
    inputs = tokenizer(palavra, return_tensors='pt')
    with torch.no_grad():
        outputs = model_base.bert(**inputs)
        return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()

def obter_vetor_base_com_grad(palavra): # atualizar os gradientes e permitir o backpropagation
    """Vetor usando BERTimbau base - com gradientes ativados"""
    inputs = tokenizer(palavra, return_tensors='pt')
    outputs = model_base.bert(**inputs)
    # .detach() remove os gradientes antes de converter para numpy
    return outputs.last_hidden_state.mean(dim=1).squeeze().detach().numpy()

def obter_vetor_ft(palavra):
    """Vetor usando BERTimbau base"""
    inputs = tokenizer(palavra, return_tensors='pt')
    with torch.no_grad():
        outputs = model_ft.bert(**inputs)
        return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()

def obter_vetor_ft_com_grad(palavra): # atualizar os gradientes e permitir o backpropagation
    """Vetor usando BERTimbau base - com gradientes ativados"""
    inputs = tokenizer(palavra, return_tensors='pt')
    outputs = model_ft.bert(**inputs)
    # .detach() remove os gradientes antes de converter para numpy
    return outputs.last_hidden_state.mean(dim=1).squeeze().detach().numpy()
# ============================================
# 7. FUNÇÃO PARA PLOTAR VETORES 3D
# ============================================
def plotar_vetores_3d(vetores, palavras, titulo="Vetores no Espaço 3D"):
    """Plota vetores de embedding no espaço 3D usando PCA"""
    if len(vetores) < 2:
        print("  Necessário pelo menos 2 vetores")
        return None

    pca = PCA(n_components=3, random_state=42)
    coords = pca.fit_transform(vetores)

    fig = go.Figure()

    fig.add_trace(go.Scatter3d(
        x=coords[:, 0],
        y=coords[:, 1],
        z=coords[:, 2],
        mode='markers+text',
        marker=dict(size=15, color=list(range(len(palavras))), colorscale='Viridis'),
        text=palavras,
        textposition="top center",
        textfont=dict(size=14)
    ))

    fig.update_layout(
        title=titulo,
        scene=dict(
            xaxis_title=f"PC1 ({pca.explained_variance_ratio_[0]:.1%})",
            yaxis_title=f"PC2 ({pca.explained_variance_ratio_[1]:.1%})",
            zaxis_title=f"PC3 ({pca.explained_variance_ratio_[2]:.1%})"
        ),
        width=900,
        height=700
    )
    fig.show()
    return coords

# ============================================
# 8. FUNÇÃO PARA CALCULAR SIMILARIDADES
# ============================================
def calcular_similaridades(vetor_alvo, tokens, obter_vetor_func, top_n=10):
    """Calcula similaridade entre vetor e todos os tokens"""
    resultados = []
    tokens_unicos = list(tokens.keys())

    for token in tokens_unicos:  # Não limita para performance, pega todos
        try:
            vetor_token = obter_vetor_func(token)
            sim = cosine_similarity([vetor_alvo], [vetor_token])[0][0]
            resultados.append((token, sim))
        except:
            continue

    resultados.sort(key=lambda x: x[1], reverse=True)
    return resultados[:top_n]

# ============================================
# 9. FUNÇÃO PARA PLOTAR VIZINHOS 3D (ATIVIDADE 2)
# ============================================
def plotar_vizinhos_3d(palavra_central, palavras_vizinhas, scores, titulo=None):
    """Plota a palavra central e seus vizinhos no espaço 3D"""

    # Coletar vetores
    todas_palavras = [palavra_central] + palavras_vizinhas[:8]
    vetores = []

    for p in todas_palavras:
        try:
            vetor = obter_vetor_base(p)
            vetores.append(vetor)
        except:
            print(f"  Erro ao obter vetor para: {p}")
            return None

    # Reduzir para 3D com PCA
    pca = PCA(n_components=3, random_state=42)
    coords = pca.fit_transform(vetores)

    fig = go.Figure()

    # Vizinhos
    for i, (palavra, score) in enumerate(zip(palavras_vizinhas[:8], scores[:8])):
        fig.add_trace(go.Scatter3d(
            x=[coords[i+1, 0]],
            y=[coords[i+1, 1]],
            z=[coords[i+1, 2]],
            mode='markers+text',
            marker=dict(size=10, color='lightblue'),
            text=[f"{palavra}<br>(sim: {score:.3f})"],
            textposition="top center",
            name=f"Vizinho: {palavra}"
        ))

    # Palavra central
    fig.add_trace(go.Scatter3d(
        x=[coords[0, 0]],
        y=[coords[0, 1]],
        z=[coords[0, 2]],
        mode='markers+text',
        marker=dict(size=10, color='red'),
        #text=[f"* {palavra_central} *"],
        text=[""],
        textposition="top center",
        textfont=dict(size=10, color='darkred'),
        name="Palavra Central"
    ))

    fig.update_layout(
        title=titulo or f"Vizinhos semânticos de '{palavra_central}'",
        scene=dict(
            xaxis_title=f"PC1 ({pca.explained_variance_ratio_[0]:.1%})",
            yaxis_title=f"PC2 ({pca.explained_variance_ratio_[1]:.1%})",
            zaxis_title=f"PC3 ({pca.explained_variance_ratio_[2]:.1%})"
        ),
        width=900,
        height=700,
        showlegend=True
    )
    fig.show()


6. FUNÇÕES PARA EXTRAIR VETORES


In [ ]:
# ============================================
# 10. ATIVIDADE 1: VETORES GERADOS
# ============================================
print("\n" + "="*60)
print("ATIVIDADE 1: VETORES GERADOS")
print("="*60)

palavras_atv1 = ["modelos", "linguagem", substantivo_top, verbo_top]
vetores_base_list = []
vetores_base_dict = {}

print("\n Vetores do modelo BASE:")
for p in palavras_atv1:
    vetor = obter_vetor_base(p)
    vetores_base_list.append(vetor)
    vetores_base_dict[p] = vetor
    print(f"\n  {p}:")
    print(f"    Primeiras 10 dim: {np.round(vetor[:10], 4)}")
    print(f"    Norma: {np.linalg.norm(vetor):.4f}")

print("\n Gráfico 3D - BASE:")
plotar_vetores_3d(vetores_base_list, palavras_atv1, "Vetores BERTimbau BASE")


ATIVIDADE 1: VETORES GERADOS

 Vetores do modelo BASE:

  modelos:
    Primeiras 10 dim: [ 0.2987 -0.0636  0.7451  0.1167 -0.0803 -0.0741  0.0614 -0.0242  0.5377
  0.4543]
    Norma: 7.4335

  linguagem:
    Primeiras 10 dim: [ 0.1415  0.1888  0.901   0.2344  0.1286 -0.0201 -0.0015 -0.2504  0.2351
  0.7475]
    Norma: 7.4939

  anotacão:
    Primeiras 10 dim: [ 0.2375 -0.1644  0.3392  0.0567  0.6791  0.1647  0.0035 -0.3438  0.3239
  0.3549]
    Norma: 6.8008

  pode:
    Primeiras 10 dim: [ 1.621e-01 -4.210e-02  7.712e-01  1.540e-01  1.930e-02  2.163e-01
 -7.780e-02  2.000e-04  2.713e-01  6.665e-01]
    Norma: 7.1346

 Gráfico 3D - BASE:


array([[ 3.69958893, -0.54574825,  0.63735784],
       [-1.10880959,  3.29271056,  1.17514959],
       [-2.06850142, -2.73916122,  1.2944943 ],
       [-0.52227792, -0.0078011 , -3.10700173]])

In [ ]:
# ============================================
# 10. ATIVIDADE 2: TERMOS MAIS SIMILARES
# ============================================
# Similaridade com modelo_base - Sem finetune
print("\n" + "="*60)
print("ATIVIDADE 2: TERMOS MAIS SIMILARES - SEM FINE TUNE")
print("="*60)

for p in palavras_atv1:
    print(f"\n  Similares a '{p}':")
    similares = calcular_similaridades(vetores_base_dict[p], freq_tokens, obter_vetor_base_com_grad, top_n=5)
    for i, (token, score) in enumerate(similares, 1):
        print(f"    {i}. {token} ({score:.4f})")

    # Gerar gráfico 3D dos vizinhos para esta palavra
    palavras_viz = [s[0] for s in similares]
    scores_viz = [s[1] for s in similares]
    plotar_vizinhos_3d(p, palavras_viz, scores_viz, f"Vizinhos semânticos de '{p}'")


ATIVIDADE 2: TERMOS MAIS SIMILARES - SEM FINE TUNE

  Similares a 'modelos':
    1. modelos (1.0000)
    2. modelo (0.8694)
    3. tamanhos (0.8569)
    4. estilos (0.8466)
    5. formas (0.8343)



  Similares a 'linguagem':
    1. linguagem (1.0000)
    2. língua (0.8667)
    3. lógica (0.8604)
    4. linguagens (0.8488)
    5. inteligência (0.8200)



  Similares a 'anotacão':
    1. anotacão (1.0000)
    2. anotacãode (0.9452)
    3. anotacão- (0.9399)
    4. anotacões (0.9371)
    5. anota (0.8860)



  Similares a 'pode':
    1. pode (1.0000)
    2. devem (0.9662)
    3. deve (0.9647)
    4. podem (0.9623)
    5. poderia (0.9620)


In [ ]:
print("\n" + "="*60)
print("ATIVIDADE 2: TERMOS MAIS SIMILARES - COM FINE TUNE")
print("="*60)

for p in palavras_atv1:
    print(f"\n  Similares a '{p}':")
    similares = calcular_similaridades(vetores_base_dict[p], freq_tokens, obter_vetor_ft_com_grad, top_n=5)
    for i, (token, score) in enumerate(similares, 1):
        print(f"    {i}. {token} ({score:.4f})")

    # Gerar gráfico 3D dos vizinhos para esta palavra
    palavras_viz = [s[0] for s in similares]
    scores_viz = [s[1] for s in similares]
    plotar_vizinhos_3d(p, palavras_viz, scores_viz, f"Vizinhos semânticos de '{p}'")


ATIVIDADE 2: TERMOS MAIS SIMILARES - COM FINE TUNE

  Similares a 'modelos':
    1. modelos (0.9214)
    2. modelo (0.7819)
    3. linhas (0.7662)
    4. marcas (0.7652)
    5. estilos (0.7628)



  Similares a 'linguagem':
    1. linguagem (0.8778)
    2. linguística (0.7987)
    3. lógica (0.7916)
    4. sintática.4 (0.7806)
    5. escrita (0.7762)



  Similares a 'anotacão':
    1. anotacão (0.9123)
    2. anotacões (0.8550)
    3. anota (0.8404)
    4. anotacão- (0.8383)
    5. aanotacão (0.8274)



  Similares a 'pode':
    1. indicar (0.9089)
    2. mencionado (0.9007)
    3. colocando (0.8996)
    4. podemos (0.8993)
    5. estejam (0.8986)


In [ ]:
import torch
import numpy as np
from collections import Counter
import plotly.express as px
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

# Initialize the tokenizer again in this cell to ensure it's defined.
tokenizer = BertTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased')

# ============================================
# ATIVIDADE 3: CLASSIFICAÇÃO DE ESTILOS COM BERTIMBAU FINE-TUNED
# Com dataset expandido (50+ exemplos por classe)
# ============================================

print("\n" + "="*60)
print("ATIVIDADE 3: CLASSIFICAÇÃO DE ESTILOS COM BERTIMBAU")
print("="*60)

from transformers import BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
import torch
import numpy as np
from collections import Counter
import plotly.express as px
import pandas as pd
from sklearn.model_selection import train_test_split

# ==========================================
# 1. DATASET PARA CLASSIFICAÇÃO
# ==========================================

class StyleDataset(Dataset):
    def __init__(self, textos, labels, tokenizer, max_length=256):
        self.textos = textos
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.label_map = {"academico": 0, "narrativo": 1, "descritivo": 2}

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.textos[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.label_map[self.labels[idx]], dtype=torch.long)
        }

# ==========================================
# 2. DATASET EXPANDIDO (50+ exemplos por classe)
# ==========================================

print("\n Criando dataset expandido com 50+ exemplos por estilo...")

# ==========================================
# ESTILO ACADÊMICO (50 exemplos)
# Características: voz passiva, terceira pessoa, impessoalidade
# ==========================================

textos_academico = [
    # Voz passiva básica
    "Observou-se que os resultados apresentam significância estatística.",
    "Foi verificado que os modelos baseados em transformer superam abordagens anteriores.",
    "Conclui-se que a metodologia empregada demonstra eficácia na tarefa proposta.",
    "Os dados foram coletados e analisados estatisticamente segundo protocolos estabelecidos.",
    "Realizou-se uma análise detalhada dos componentes principais do modelo.",
    "Verificou-se uma correlação significativa entre as variáveis analisadas.",
    "Pode-se observar que os resultados obtidos são consistentes com a literatura.",
    "Nota-se uma tendência clara de melhoria no desempenho dos classificadores.",
    "Foi demonstrado que a abordagem proposta é eficaz para o problema em questão.",
    "Conduziu-se um experimento controlado para avaliar o impacto das configurações.",

    # Impessoalidade e formalidade
    "É importante ressaltar que os resultados devem ser interpretados com cautela.",
    "Considera-se que a amostra utilizada é representativa da população estudada.",
    "Entende-se que os achados contribuem significativamente para a área de conhecimento.",
    "Salienta-se a necessidade de replicação dos experimentos em diferentes contextos.",
    "Destaca-se a relevância dos resultados obtidos para aplicações práticas.",
    "Ressalta-se que as limitações do estudo não comprometem as conclusões principais.",
    "Argumenta-se que os modelos neurais apresentam vantagens sobre métodos tradicionais.",
    "Sugere-se que pesquisas futuras investiguem a generalização dos resultados.",
    "Infere-se dos dados que existe uma relação causal entre as variáveis estudadas.",
    "Depreende-se da análise que os resultados são robustos a diferentes configurações.",

    # Análise e metodologia
    "A análise estatística foi realizada utilizando o software R versão 4.0.",
    "Os experimentos foram conduzidos em ambiente controlado com temperatura constante.",
    "A métrica de avaliação utilizada foi a acurácia balanceada devido ao desbalanceamento.",
    "O conjunto de dados foi dividido em treino (70%), validação (15%) e teste (15%).",
    "A significância estatística foi avaliada utilizando o teste t de Student (p<0,05).",
    "Os intervalos de confiança foram calculados utilizando o método bootstrap com 1000 replicações.",
    "A validação cruzada com 10 folds foi empregada para avaliar a estabilidade do modelo.",
    "O pré-processamento incluiu remoção de stopwords, stemming e normalização de caixa.",
    "A matriz de confusão revelou que o modelo apresenta alta precisão e recall.",
    "A curva ROC apresentou AUC de 0,95, indicando excelente poder discriminativo.",

    # Resultados e conclusões
    "Os resultados indicam que a hipótese nula foi rejeitada (p<0,001).",
    "A análise de variância mostrou diferenças significativas entre os grupos experimentais.",
    "O coeficiente de correlação de Pearson foi de 0,87 (p<0,01), indicando forte correlação.",
    "Os modelos baseados em BERT superaram significativamente as abordagens baseline (p<0,05).",
    "A acurácia média do modelo proposto foi de 94,5% (DP=1,2%) nos dados de teste.",
    "Os resultados sugerem que o método proposto é robusto a ruídos nos dados de entrada.",
    "A análise de sensibilidade mostrou que o modelo é estável para variações nos parâmetros.",
    "Os experimentos de ablação confirmaram a importância de cada componente do sistema.",
    "A validação externa em um corpus independente confirmou a generalização dos resultados.",
    "Conclui-se que a abordagem proposta é promissora para aplicações em PLN.",

    # Variações adicionais
    "Foi observada uma melhoria consistente de 15% em relação ao estado da arte.",
    "Realizou-se uma busca sistemática na literatura para identificar trabalhos relacionados.",
    "A métrica F1 foi escolhida como medida principal devido ao desbalanceamento das classes.",
    "O modelo foi treinado por 50 épocas, com early stopping baseado na perda de validação.",
    "A curva de aprendizado mostrou convergência após aproximadamente 30 épocas de treinamento.",
    "A análise post-hoc revelou que os resultados são robustos a diferentes sementes aleatórias.",
    "O poder estatístico do estudo foi calculado como 0,95 para detectar diferenças de 10%.",
    "O tamanho do efeito (Cohen's d) foi calculado como 0,85, indicando efeito grande.",
    "Os resultados foram validados utilizando o método de Bonferroni para correção de múltiplas comparações.",
    "A análise de subgrupos revelou que os resultados são consistentes entre diferentes faixas etárias."
]

# ==========================================
# ESTILO NARRATIVO (50 exemplos)
# Características: primeira pessoa do plural, fluidez, reflexividade
# ==========================================

textos_narrativo = [
    # Primeira pessoa - análise
    "Analisamos os dados coletados durante o experimento e percebemos padrões interessantes.",
    "Exploramos diferentes configurações do modelo e encontramos resultados promissores.",
    "Investigamos a influência do contexto e observamos que ele é fundamental para o desempenho.",
    "Avaliamos nossa abordagem em múltiplos corpora e verificamos sua eficácia.",
    "Implementamos um novo algoritmo que, em nossos testes, superou as alternativas existentes.",
    "Comparamos nossa metodologia com técnicas state-of-the-art e obtivemos resultados superiores.",
    "Testamos nossa hipótese em diferentes cenários e confirmamos nossas expectativas iniciais.",
    "Validamos nossa abordagem com especialistas da área e recebemos feedback positivo.",
    "Aplicamos o modelo proposto em problemas reais e obtivemos resultados encorajadores.",
    "Desenvolvemos uma solução que atende às necessidades identificadas em nossa pesquisa.",

    # Reflexões e interpretações
    "Acreditamos que nossos resultados abrem novas perspectivas para pesquisas futuras.",
    "Consideramos que a abordagem desenvolvida representa um avanço significativo na área.",
    "Pensamos que as limitações identificadas não comprometem a validade das conclusões.",
    "Entendemos que ainda há espaço para melhorias, especialmente no pré-processamento.",
    "Refletimos sobre as implicações éticas do uso de modelos de linguagem em larga escala.",
    "Acreditamos que nossa contribuição pode beneficiar outros pesquisadores da comunidade.",
    "Consideramos importante compartilhar nosso código e dados para promover reprodutibilidade.",
    "Pensamos que a interpretabilidade dos modelos é um desafio crucial a ser enfrentado.",
    "Acreditamos que trabalhos futuros devem investigar a aplicação em outros domínios.",
    "Refletimos sobre como nossa abordagem se alinha com teorias linguísticas estabelecidas.",

    # Descobertas e observações
    "Percebemos que o desempenho do modelo varia significativamente com o tamanho do corpus.",
    "Observamos que a remoção de stopwords teve impacto modesto nos resultados finais.",
    "Notamos que modelos pré-treinados em português superam aqueles treinados em multilíngue.",
    "Verificamos que o fine-tuning com poucos exemplos já produz resultados razoáveis.",
    "Constamos que a normalização dos dados é crucial para a estabilidade do treinamento.",
    "Detectamos que certos tipos de erro são sistemáticos e merecem investigação adicional.",
    "Identificamos padrões que sugerem a necessidade de uma abordagem híbrida.",
    "Descobrimos que o contexto local é mais importante que o contexto global para esta tarefa.",
    "Confirmamos nossa hipótese de que a arquitetura proposta é mais eficiente.",
    "Validamos empiricamente as vantagens teóricas esperadas do nosso método.",

    # Propostas e perspectivas
    "Propomos uma nova arquitetura que combina o melhor de diferentes abordagens.",
    "Sugerimos que pesquisas futuras investiguem a aplicação em dados multimodais.",
    "Recomendamos a adoção de nossas diretrizes para anotação de corpus.",
    "Defendemos que a comunidade adote práticas mais rigorosas de avaliação.",
    "Apresentamos uma análise detalhada que esperamos ser útil para outros pesquisadores.",
    "Compartilhamos nossas implementações para facilitar a reprodução dos experimentos.",
    "Disponibilizamos nossos dados anotados para promover avanços na área.",
    "Convidamos a comunidade a explorar as muitas questões em aberto identificadas.",
    "Encaminhamos nossa pesquisa para aplicações práticas em sistemas reais.",
    "Visualizamos um futuro onde modelos como este serão ubíquos em aplicações de linguagem.",

    # Narrativa processual
    "Começamos nossa pesquisa com uma revisão sistemática da literatura especializada.",
    "Selecionamos cuidadosamente os conjuntos de dados que melhor representam o domínio.",
    "Projetamos experimentos para testar cada uma de nossas hipóteses de pesquisa.",
    "Coletamos dados de múltiplas fontes para garantir diversidade e representatividade.",
    "Processamos os dados utilizando pipelines que desenvolvemos especificamente para este fim.",
    "Treinamos nossos modelos em infraestrutura de GPU de última geração.",
    "Avaliamos os resultados utilizando métricas estabelecidas pela comunidade.",
    "Interpretamos os achados à luz das teorias existentes e de nossas contribuições.",
    "Documentamos todo o processo para garantir transparência e reprodutibilidade.",
    "Divulgamos nossos resultados em conferências e periódicos de alto impacto."
]

# ==========================================
# ESTILO DESCRITIVO (50 exemplos)
# Características: frases curtas, dados, sequência lógica, objetividade
# ==========================================

textos_descritivo = [
    # Dados e métricas
    "O corpus contém 10.000 documentos. Cada documento possui 512 tokens.",
    "A acurácia foi de 94,5%. O desvio padrão é 0,03. O intervalo de confiança é 95%.",
    "Precisão: 97,3%. Recall: 94,1%. F1: 95,7%. AUC: 0,96.",
    "Média: 85,4. Mediana: 87,2. Variância: 12,5. Desvio: 3,54.",
    "Experimento A: n=1000, média=75,2. Experimento B: n=1000, média=78,4.",
    "Tempo de treinamento: 2h30min. Número de parâmetros: 110M. Memória: 12GB.",
    "Batch size: 32. Learning rate: 2e-5. Épocas: 10. Dropout: 0,1.",
    "CPU: Intel i7-10700. GPU: NVIDIA RTX 3080. RAM: 32GB. Tempo: 45min.",
    "Erro quadrático médio: 0,023. Erro absoluto médio: 0,112. R²: 0,94.",
    "Sensibilidade: 0,89. Especificidade: 0,92. Valor preditivo positivo: 0,91.",

    # Sequências lógicas
    "Etapa 1: pré-processamento. Etapa 2: tokenização. Etapa 3: classificação.",
    "Primeiro, carregar dados. Segundo, normalizar. Terceiro, treinar. Quarto, testar.",
    "Passo 1: coletar corpus. Passo 2: anotar dados. Passo 3: treinar modelo.",
    "Fase 1: preparação. Fase 2: experimentação. Fase 3: análise. Fase 4: documentação.",
    "1º extrair features. 2º normalizar. 3º aplicar PCA. 4º classificar. 5º avaliar.",
    "Inicialmente, carregar bibliotecas. Em seguida, preparar dados. Depois, treinar modelo.",
    "Primeira etapa: coleta. Segunda etapa: limpeza. Terceira etapa: modelagem.",
    "Nível 1: básico. Nível 2: intermediário. Nível 3: avançado.",
    "Dia 1: planejamento. Dia 2: implementação. Dia 3: testes. Dia 4: documentação.",
    "Sprint 1: análise. Sprint 2: desenvolvimento. Sprint 3: validação.",

    # Resultados numéricos
    "Resultados: 85% de acurácia. 78% de precisão. 82% de recall.",
    "Classificação: classe A (45%), classe B (32%), classe C (23%).",
    "Distribuição: treino (60%), validação (20%), teste (20%).",
    "Estatísticas: mínimo=12, máximo=98, média=54,3, mediana=51,0.",
    "Correlações: X1 (0,45), X2 (0,67), X3 (0,23), X4 (0,89).",
    "Tabela 1: resultados por configuração. Tabela 2: análise de erro.",
    "Figura 1: curva ROC (AUC=0,94). Figura 2: matriz de confusão.",
    "Comparação: método A (92%), método B (88%), método C (85%).",
    "Aumento de 15% em relação ao baseline. Redução de 23% no erro.",
    "Tempo de inferência: 0,23ms por exemplo. Throughput: 4340 exemplos/segundo.",

    # Instruções e procedimentos
    "Execute: python train.py --config config.yaml --epochs 50.",
    "Comando: pip install -r requirements.txt. Depois: python main.py.",
    "Instalação: conda create -n env python=3.8. conda activate env.",
    "Configuração: edite o arquivo .env com suas credenciais.",
    "Uso: from modelo import Classificador; c = Classificador()",
    "Parâmetros: --input data/ --output results/ --model bert-base",
    "Logs: verificar arquivo logs/experiment_20241215.log",
    "Erro: ValueError: incompatible shapes. Solução: verificar dimensões.",
    "Aviso: deprecation warning. Atualizar para versão 2.0.",
    "Nota: resultados podem variar com diferentes sementes aleatórias.",

    # Variações descritivas
    "3 experimentos. 5 repetições. 2 condições. Total: 30 medições.",
    "Tamanho: pequeno (<1K), médio (1K-10K), grande (>10K).",
    "Temperatura: 25°C. Umidade: 45%. Pressão: 1013 hPa.",
    "Duração: curta (<1h), média (1-4h), longa (>4h).",
    "Custo: baixo (<$100), médio ($100-$1000), alto (>$1000).",
    "Prioridade: alta (1), média (2), baixa (3). Status: pendente (0), concluído (1).",
    "Versão: 1.0.0 (estável), 2.0.0-beta (experimental).",
    "Licença: MIT (código aberto), CC-BY-4.0 (dados).",
    "Formato: JSON (dados), YAML (configuração), Markdown (documentação).",
    "Encoding: UTF-8. Line ending: LF. Indentação: 2 espaços."
]

# Garantir que todos os arrays tenham o mesmo tamanho
print(f"Acadêmico: {len(textos_academico)} exemplos")
print(f"Narrativo: {len(textos_narrativo)} exemplos")
print(f"Descritivo: {len(textos_descritivo)} exemplos")

# Combinar todos os textos
todos_textos = textos_academico + textos_narrativo + textos_descritivo
todos_labels = (["academico"] * len(textos_academico) +
                ["narrativo"] * len(textos_narrativo) +
                ["descritivo"] * len(textos_descritivo))

print(f"\nTotal de exemplos anotados: {len(todos_textos)}")
print(f"   • Acadêmico: {len(textos_academico)} exemplos")
print(f"   • Narrativo: {len(textos_narrativo)} exemplos")
print(f"   • Descritivo: {len(textos_descritivo)} exemplos")

# ==========================================
# 3. DIVIDIR EM TREINO E VALIDAÇÃO
# ==========================================

train_texts, val_texts, train_labels, val_labels = train_test_split(
    todos_textos, todos_labels, test_size=0.2, random_state=42, stratify=todos_labels
)

print(f"\nDivisão dos dados:")
print(f"   Treino: {len(train_texts)} exemplos")
print(f"   Validação: {len(val_texts)} exemplos")
print(f"   Proporção treino/validação: 80/20")

# ==========================================
# 4. CRIAR DATASETS
# ==========================================

train_dataset = StyleDataset(train_texts, train_labels, tokenizer)
val_dataset = StyleDataset(val_texts, val_labels, tokenizer)

# ==========================================
# 5. CARREGAR MODELO PARA CLASSIFICAÇÃO
# ==========================================

model_classifier = BertForSequenceClassification.from_pretrained(
    'neuralmind/bert-base-portuguese-cased',
    num_labels=3
)

print(f"\nModelo BERTimbau carregado para classificação")
print(f"   Classes: {model_classifier.config.num_labels} (acadêmico, narrativo, descritivo)")
print(f"   Tamanho do vocabulário: {model_classifier.config.vocab_size}")

# ==========================================
# 6. CONFIGURAR TREINAMENTO
# ==========================================

training_args = TrainingArguments(
    output_dir="./classificador_estilos",
    num_train_epochs=3,
    per_device_train_batch_size=8,  # Batch maior
    per_device_eval_batch_size=8,
    save_strategy="no",
    logging_steps=10,
    report_to="none"
)

trainer = Trainer(
    model=model_classifier,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# ==========================================
# 7. EXECUTAR FINE-TUNING
# ==========================================

print("\nIniciando fine-tuning do BERTimbau...")
print(f"   Dados de treino: {len(train_dataset)} exemplos")
print(f"   Dados de validação: {len(val_dataset)} exemplos")
print(f"   Épocas: {training_args.num_train_epochs}")
print(f"   Batch size: {training_args.per_device_train_batch_size}")
print("\n   Executar o fine-tuning")


trainer.train()  # Executar o fine-tuning

print("\nFine-tuning configurado com sucesso!")

# ==========================================
# 8. FUNÇÃO PARA CLASSIFICAR
# ==========================================

def classificar_texto(texto):
    """Classifica um texto usando o modelo fine-tuned"""
    inputs = tokenizer(texto, return_tensors="pt", truncation=True, padding=True, max_length=256)
    with torch.no_grad():
        outputs = model_classifier(**inputs)

    pred = torch.argmax(outputs.logits, dim=1).item()
    probs = torch.softmax(outputs.logits, dim=1).squeeze().numpy()

    labels_map = {0: "academico", 1: "narrativo", 2: "descritivo"}

    return {
        'estilo': labels_map[pred],
        'probabilidades': {
            'academico': float(probs[0]),
            'narrativo': float(probs[1]),
            'descritivo': float(probs[2])
        }
    }

# ==========================================
# 9. TESTAR O CLASSIFICADOR
# ==========================================

print("\n" + "="*60)
print("TESTANDO O CLASSIFICADOR")
print("="*60)

testes = [
    ("Observou-se que os resultados são estatisticamente significativos (p<0,05).", "academico"),
    ("Analisamos os dados cuidadosamente e percebemos padrões muito interessantes.", "narrativo"),
    ("Acurácia: 94,5%. Precisão: 93,2%. Recall: 91,8%. F1: 92,3%.", "descritivo"),
    ("Foi demonstrado que o modelo proposto supera as abordagens baseline.", "academico"),
    ("Acreditamos que nossa contribuição pode beneficiar toda a comunidade científica.", "narrativo"),
    ("Passo 1: carregar. Passo 2: processar. Passo 3: analisar. Passo 4: concluir.", "descritivo")
]

for texto, esperado in testes:
    resultado = classificar_texto(texto)
    print(f"\nTexto: {texto[:80]}...")
    print(f"   Esperado: {esperado}")
    print(f"   Classificado: {resultado['estilo']}")
    print(f"   Confiança: {max(resultado['probabilidades'].values()):.2%}")

# ==========================================
# 10. CLASSIFICAR ARTIGOS DO CORPUS
# ==========================================

print("\n" + "="*60)
print("CLASSIFICANDO ARTIGOS DO CORPUS STIL 2023")
print("="*60)

resultados = []

for i, artigo in enumerate(articles, 1):
    titulo = artigo.get("titulo", f"Artigo {i}")
    texto = artigo.get("artigo_completo", "")

    if not texto or len(texto) < 200:
        continue

    texto_classificar = titulo + ". " + texto
    resultado = classificar_texto(texto_classificar)

    resultados.append({
        'id': i,
        'titulo': titulo,
        'estilo': resultado['estilo'],
        'probabilidades': resultado['probabilidades']
    })

    if i % 10 == 0:
        print(f"   Processados {i} de {len(articles)} artigos...")

print(f"\n{len(resultados)} artigos classificados!")

# ==========================================
# 11. EXIBIR RESULTADOS
# ==========================================

print("\n" + "="*60)
print("RESULTADOS DA CLASSIFICAÇÃO")
print("="*60)

print(f"\n{'#':3} | {'ESTILO':12} | {'ACADÊMICO':10} | {'NARRATIVO':10} | {'DESCRITIVO':10} | TÍTULO")
print("-" * 95)

for r in resultados[:20]:
    p = r['probabilidades']
    estilo_display = r['estilo'].upper()
    titulo_resumido = r['titulo'][:45] + "..." if len(r['titulo']) > 45 else r['titulo']
    print(f"{r['id']:3} | {estilo_display:12} | {p['academico']:9.2%} | {p['narrativo']:9.2%} | {p['descritivo']:9.2%} | {titulo_resumido}")

# ==========================================
# 12. ESTATÍSTICAS FINAIS
# ==========================================

contagem = Counter([r['estilo'] for r in resultados])

print("\n" + "="*60)
print("ESTATÍSTICAS FINAIS")
print("="*60)
print(f" ACADÊMICO:  {contagem.get('academico', 0)} artigos")
print(f" NARRATIVO:  {contagem.get('narrativo', 0)} artigos")
print(f" DESCRITIVO: {contagem.get('descritivo', 0)} artigos")
print(f"\n Total: {len(resultados)} artigos analisados")

# ==========================================
# 13. GRÁFICO
# ==========================================

df_plot = pd.DataFrame({
    'Estilo': ['Acadêmico', 'Narrativo', 'Descritivo'],
    'Quantidade': [contagem.get('academico', 0), contagem.get('narrativo', 0), contagem.get('descritivo', 0)]
})

fig = px.bar(df_plot, x='Estilo', y='Quantidade',
             title='Distribuição de Estilos de Escrita (BERTimbau Fine-tuned)',
             color='Estilo',
             color_discrete_map={'Acadêmico': '#2E86AB', 'Narrativo': '#A23B72', 'Descritivo': '#F18F01'},
             text='Quantidade')
fig.update_layout(width=600, height=450)
fig.show()

# ==========================================
# 14. JUSTIFICATIVA
# ==========================================

print("\n" + "="*60)
print("JUSTIFICATIVA DA CLASSIFICAÇÃO")
print("="*60)

print(f"""
┌─────────────────────────────────────────────────────────────────────────────┐
│                    RELATÓRIO DA CLASSIFICAÇÃO DE ESTILOS                    │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│  DATASET DE TREINO                                                           │
│    • Total de exemplos anotados: {len(todos_textos)}                         │
│    • Acadêmico: {len(textos_academico)} exemplos (voz passiva, impessoalidade)│
│    • Narrativo: {len(textos_narrativo)} exemplos (1ª pessoa, reflexividade)  │
│    • Descritivo: {len(textos_descritivo)} exemplos (dados, sequência lógica) │
│                                                                              │
│  RESULTADOS DA CLASSIFICAÇÃO                                                 │
│    • Artigos classificados: {len(resultados)}                                │
│    • Acadêmico: {contagem.get('academico', 0)} artigos ({contagem.get('academico', 0)/len(resultados)*100:.1f}%) │
│    • Narrativo: {contagem.get('narrativo', 0)} artigos ({contagem.get('narrativo', 0)/len(resultados)*100:.1f}%) │
│    • Descritivo: {contagem.get('descritivo', 0)} artigos ({contagem.get('descritivo', 0)/len(resultados)*100:.1f}%) │
│                                                                              │
│  CONFIABILIDADE DO MODELO                                                    │
│    • Dataset balanceado com {len(todos_textos)} exemplos                     │
│    • Fine-tuning com {training_args.num_train_epochs} épocas                 │
│    • Validação em {len(val_dataset)} exemplos (20% dos dados)                │
│    • Acurácia esperada: >85% nos dados de validação                          │
│                                                                              │
└─────────────────────────────────────────────────────────────────────────────┘
""")


ATIVIDADE 3: CLASSIFICAÇÃO DE ESTILOS COM BERTIMBAU

 Criando dataset expandido com 50+ exemplos por estilo...
Acadêmico: 50 exemplos
Narrativo: 50 exemplos
Descritivo: 50 exemplos

Total de exemplos anotados: 150
   • Acadêmico: 50 exemplos
   • Narrativo: 50 exemplos
   • Descritivo: 50 exemplos

Divisão dos dados:
   Treino: 120 exemplos
   Validação: 30 exemplos
   Proporção treino/validação: 80/20


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th


Modelo BERTimbau carregado para classificação
   Classes: 3 (acadêmico, narrativo, descritivo)
   Tamanho do vocabulário: 29794

Iniciando fine-tuning do BERTimbau...
   Dados de treino: 120 exemplos
   Dados de validação: 30 exemplos
   Épocas: 3
   Batch size: 8

   Executar o fine-tuning


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning:

'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.



Step,Training Loss
10,0.849846
20,0.319755
30,0.092884
40,0.040483



Fine-tuning configurado com sucesso!

TESTANDO O CLASSIFICADOR

Texto: Observou-se que os resultados são estatisticamente significativos (p<0,05)....
   Esperado: academico
   Classificado: academico
   Confiança: 96.89%

Texto: Analisamos os dados cuidadosamente e percebemos padrões muito interessantes....
   Esperado: narrativo
   Classificado: narrativo
   Confiança: 96.90%

Texto: Acurácia: 94,5%. Precisão: 93,2%. Recall: 91,8%. F1: 92,3%....
   Esperado: descritivo
   Classificado: descritivo
   Confiança: 97.36%

Texto: Foi demonstrado que o modelo proposto supera as abordagens baseline....
   Esperado: academico
   Classificado: academico
   Confiança: 96.85%

Texto: Acreditamos que nossa contribuição pode beneficiar toda a comunidade científica....
   Esperado: narrativo
   Classificado: narrativo
   Confiança: 95.63%

Texto: Passo 1: carregar. Passo 2: processar. Passo 3: analisar. Passo 4: concluir....
   Esperado: descritivo
   Classificado: descritivo
   Confiança: 97.55%



JUSTIFICATIVA DA CLASSIFICAÇÃO

┌─────────────────────────────────────────────────────────────────────────────┐
│                    RELATÓRIO DA CLASSIFICAÇÃO DE ESTILOS                    │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│  DATASET DE TREINO                                                           │
│    • Total de exemplos anotados: 150                         │
│    • Acadêmico: 50 exemplos (voz passiva, impessoalidade)│
│    • Narrativo: 50 exemplos (1ª pessoa, reflexividade)  │
│    • Descritivo: 50 exemplos (dados, sequência lógica) │
│                                                                              │
│  RESULTADOS DA CLASSIFICAÇÃO                                                 │
│    • Artigos classificados: 30                                │
│    • Acadêmico: 3 artigos (10.0%) │
│    • Narrativo: 6 artigos (20.0%) │
│    • Descritiv

In [ ]:
# ============================================
# GRÁFICO DE BARRAS - DISTRIBUIÇÃO DOS ESTILOS (BERTIMBAU)
# ============================================

print("\n" + "="*60)
print("GRÁFICO DE BARRAS - DISTRIBUIÇÃO DOS ESTILOS (BERTIMBAU)")
print("="*60)

# Preparar dados para o gráfico
df_plot = pd.DataFrame({
    'Estilo': ['Acadêmico', 'Narrativo', 'Descritivo'],
    'Quantidade': [
        contagem.get('academico', 0),
        contagem.get('narrativo', 0),
        contagem.get('descritivo', 0)
    ]
})

# Criar gráfico de barras com Plotly
fig = px.bar(
    df_plot,
    x='Estilo',
    y='Quantidade',
    title='Distribuição de Estilos de Escrita (BERTimbau Fine-tuned)',
    color='Estilo',
    color_discrete_map={
        'Acadêmico': '#2E86AB',    # Azul
        'Narrativo': '#A23B72',    # Roxo
        'Descritivo': '#F18F01'     # Laranja
    },
    text='Quantidade'
)

# Ajustar layout
fig.update_traces(
    textposition='auto',
    textfont_size=14
)

fig.update_layout(
    width=600,
    height=450,
    title_font_size=16,
    xaxis_title="Estilo",
    yaxis_title="Quantidade",
    showlegend=True,
    legend_title_text="Estilo"
)

# Exibir o gráfico
fig.show()

# Exibir também em texto
print(f"\nDistribuição encontrada:")
print(f"  ACADÊMICO:  {contagem.get('academico', 0)} artigos")
print(f"  NARRATIVO:  {contagem.get('narrativo', 0)} artigos")
print(f"  DESCRITIVO: {contagem.get('descritivo', 0)} artigos")


GRÁFICO DE BARRAS - DISTRIBUIÇÃO DOS ESTILOS (BERTIMBAU)



Distribuição encontrada:
  ACADÊMICO:  3 artigos
  NARRATIVO:  6 artigos
  DESCRITIVO: 21 artigos


In [ ]:
# ==========================================
# 15. MATRIZ DE CONFUSÃO (CORRIGIDA)
# ==========================================

print("\n" + "="*60)
print("MATRIZ DE CONFUSÃO DA CLASSIFICAÇÃO")
print("="*60)

from sklearn.metrics import confusion_matrix, classification_report
import plotly.figure_factory as ff
import numpy as np

# Mapeamento dos labels
label_map = {"academico": 0, "narrativo": 1, "descritivo": 2}
label_names = ["Acadêmico", "Narrativo", "Descritivo"]

# ==========================================
# 15.1 MATRIZ DE CONFUSÃO PARA DADOS DE VALIDAÇÃO
# ==========================================

print("\n Matriz de Confusão - Dados de Validação")
print("-" * 40)

# Obter predições para o conjunto de validação
predictions = trainer.predict(val_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)

# CORREÇÃO: Usar o label_map do dataset, não do trainer
true_labels = [val_dataset.label_map[label] for label in val_labels]

# Calcular matriz de confusão
cm = confusion_matrix(true_labels, pred_labels)

# Exibir matriz como tabela
print("\nMatriz de Confusão (valores absolutos):")
print("                  Predito")
print("                  Acadêmico  Narrativo  Descritivo")
print(f"Real  Acadêmico   {cm[0,0]:>9}  {cm[0,1]:>9}  {cm[0,2]:>9}")
print(f"      Narrativo   {cm[1,0]:>9}  {cm[1,1]:>9}  {cm[1,2]:>9}")
print(f"      Descritivo  {cm[2,0]:>9}  {cm[2,1]:>9}  {cm[2,2]:>9}")

# Calcular métricas por classe
print("\n Métricas por Classe:")
print("-" * 40)

for i, nome in enumerate(label_names):
    tp = cm[i, i]
    fp = sum(cm[:, i]) - tp
    fn = sum(cm[i, :]) - tp
    tn = sum(sum(cm)) - tp - fp - fn

    # Calcular métricas
    precisao = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precisao * recall) / (precisao + recall) if (precisao + recall) > 0 else 0

    print(f"\n  {nome}:")
    print(f"    Precisão: {precisao:.2%}")
    print(f"    Recall:   {recall:.2%}")
    print(f"    F1-Score: {f1:.2%}")

# Acurácia geral
acuracia = np.trace(cm) / np.sum(cm)
print(f"\n Acurácia Geral: {acuracia:.2%}")

# ==========================================
# 15.2 GRÁFICO DA MATRIZ DE CONFUSÃO (Plotly)
# ==========================================

print("\n Gráfico da Matriz de Confusão")

# Criar dataframe para o heatmap
df_cm = pd.DataFrame(
    cm,
    index=[f"Real: {n}" for n in label_names],
    columns=[f"Predito: {n}" for n in label_names]
)

# Criar heatmap com Plotly
fig = ff.create_annotated_heatmap(
    z=cm,
    x=label_names,
    y=label_names,
    annotation_text=cm,
    colorscale='Blues',
    showscale=True,
    font_colors=['white', 'black']
)

# Melhorar layout
fig.update_layout(
    title=dict(
        text="Matriz de Confusão - Classificação de Estilos",
        font=dict(size=18)
    ),
    width=600,
    height=500,
    xaxis=dict(title="Predito", side="bottom"),
    yaxis=dict(title="Real", autorange="reversed")
)

fig.show()

# ==========================================
# 15.3 RELATÓRIO DE CLASSIFICAÇÃO COMPLETO
# ==========================================

print("\n Relatório de Classificação - Dados de Validação")
print("-" * 40)

# Usar classification_report do sklearn
report = classification_report(
    true_labels,
    pred_labels,
    target_names=label_names,
    digits=4
)
print(report)

# ==========================================
# 15.4 DISTRIBUIÇÃO DOS ERROS
# ==========================================

print("\n Análise dos Erros de Classificação")
print("-" * 40)

# Para cada classe, mostrar onde os erros estão ocorrendo
for i, nome in enumerate(label_names):
    erros = []
    for j, nome_pred in enumerate(label_names):
        if i != j and cm[i, j] > 0:
            erros.append(f"  {cm[i, j]} classificados como {nome_pred}")

    if erros:
        print(f"\n  {nome}:")
        for erro in erros:
            print(f"    {erro}")
    else:
        print(f"\n  {nome}: Nenhum erro de classificação")

# ==========================================
# 15.5 MATRIZ DE CONFUSÃO NORMALIZADA (PORCENTAGENS)
# ==========================================

print("\n Matriz de Confusão Normalizada (%)")
print("-" * 40)

# Normalizar por linha (porcentagem de cada classe real)
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

print("\nMatriz de Confusão (percentuais por classe real):")
print("                  Predito")
print("                  Acadêmico  Narrativo  Descritivo")
print(f"Real  Acadêmico   {cm_percent[0,0]:>8.1f}%  {cm_percent[0,1]:>8.1f}%  {cm_percent[0,2]:>8.1f}%")
print(f"      Narrativo   {cm_percent[1,0]:>8.1f}%  {cm_percent[1,1]:>8.1f}%  {cm_percent[1,2]:>8.1f}%")
print(f"      Descritivo  {cm_percent[2,0]:>8.1f}%  {cm_percent[2,1]:>8.1f}%  {cm_percent[2,2]:>8.1f}%")

print("\n Matriz de Confusão concluída!")


MATRIZ DE CONFUSÃO DA CLASSIFICAÇÃO

 Matriz de Confusão - Dados de Validação
----------------------------------------


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning:

'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.




Matriz de Confusão (valores absolutos):
                  Predito
                  Acadêmico  Narrativo  Descritivo
Real  Acadêmico          10          0          0
      Narrativo           0         10          0
      Descritivo          1          0          9

 Métricas por Classe:
----------------------------------------

  Acadêmico:
    Precisão: 90.91%
    Recall:   100.00%
    F1-Score: 95.24%

  Narrativo:
    Precisão: 100.00%
    Recall:   100.00%
    F1-Score: 100.00%

  Descritivo:
    Precisão: 100.00%
    Recall:   90.00%
    F1-Score: 94.74%

 Acurácia Geral: 96.67%

 Gráfico da Matriz de Confusão



 Relatório de Classificação - Dados de Validação
----------------------------------------
              precision    recall  f1-score   support

   Acadêmico     0.9091    1.0000    0.9524        10
   Narrativo     1.0000    1.0000    1.0000        10
  Descritivo     1.0000    0.9000    0.9474        10

    accuracy                         0.9667        30
   macro avg     0.9697    0.9667    0.9666        30
weighted avg     0.9697    0.9667    0.9666        30


 Análise dos Erros de Classificação
----------------------------------------

  Acadêmico: Nenhum erro de classificação

  Narrativo: Nenhum erro de classificação

  Descritivo:
      1 classificados como Acadêmico

 Matriz de Confusão Normalizada (%)
----------------------------------------

Matriz de Confusão (percentuais por classe real):
                  Predito
                  Acadêmico  Narrativo  Descritivo
Real  Acadêmico      100.0%       0.0%       0.0%
      Narrativo        0.0%     100.0%       0.0%
      